In [1]:
import pandas as pd
import numpy as np

# Marotta e gli stadi in Italia

In questo python notebook cercheremo di analizzare la seguente frase del dirigente sportivo dell'Inter Marotta:

> Questa è la nota dolente, i numeri contano per fare delle riflessioni. Negli ultimi 20 anni, in Europa, sono stati costruiti 250 nuovi stadi, di questi solo 5 in Italia. Questo la dice lunga su quanto siamo fanalino di coda

Se partiamo dal presupposto che l'intento di Marotta sia quello di evidenziare una carenza di nuovi stadi in Italia, e quindi usa questa frase per provarlo, possiamo cercare di capire se ha ragione o meno usando solo la frase stessa.

## Livello 1: 250, 5, 20 e l'Europa

Il primo livello di analisi utilizza, come informazione, solo quella contenuta nella frase di Marotta, ossia 250 stadi in 20 anni in Europa e 5 in Italia. Se l'intento di Marotta e' quello di concludere che in Italia si costruiscono pochi stadi, allora possiamo calcolare se cinque nuovi stadi in Italia, rispetto ai rimanenti 245 nel resto d'Europa, siano, in maniera significativa, statisticamente molto al di sotto della media Europea. 

Dato che Marotta parla in generale di Europa, consideriamo prima tre diverse ipotesi su cosa intendesse per Europa. Dato che non specifica a cosa si possa riferire quando parla di Europa, prendiamo come prima istanza 3 diverse definizioni di Europa per calcolare quante nazioni, in totale, sono nel continente:
- Europa come EU
- Europa come continente
- Europa come UEFA

A seconda delle definizioni di Europa, cambia il numero di nazioni che ne fanno parte e questo influenza la media di nuovi stadi per ogni paese: 

$$
\overline{nuovi\_stadi} = \frac{250}{k}
$$

dove $k$ rappresenta il numero di nazioni della definizione di Europa. 

### 1.a. Europa come EU

Nel caso in cui vogliamo considerare Europa come gli attuali stati che formano l'Unione Europea, allora $k = 27$ e quindi avremo una media attesa di:

$$
\mu = \frac{250}{27} \approx 9.26
$$

In [8]:
print(f"mu = {np.round(250/27, 2)}")

mu = 9.26


L'Italia, a quanto dice Marotta, negli ultimi 20 anni ha costruito 5 stadi, quindi la differenza con la media e' di:

$$
x - \mu = 5 -9.26 = -4.26
$$

e quindi, in Italia, ha circa 4.26 in meno rispetto alla media Europea. E' abbastanza per dire che siamo sotto la media europea? Supponiamo che il numero di stadi per ogni nazione europea segua un modello di Poisson (ossia, esiste la stessa probabilita' di costruire nuovi stadi in ogni nazione in maniera indipendente), quindi ogni nuovo stadio ha probabilita' 

$$
p = \frac{1}{k} = \frac{1}{27}
$$

di essere costruito in quella nazione, allora il numero di stadi per ogni nazione risulta:

$$
X \sim \text{Binomial}(n = 250, p = 1/27)
$$

E dato che $n$ e' grande mentre $p$ e' relativamente piccolo, allora possiamo usare un'approssimazione di Poisson:

$$
X \sim Poisson(\lambda = np = 9.26)
$$

Possiamo ora rispondere alla domanda: qual'e' la probabilita' di osservare la costruzione di 5 o meno di 5 stadi:

$$
P( X \le 5 | \lambda = 9.26)
$$

In [3]:
from scipy.stats import poisson

n = 250
p = 1/27
lambda_ = n*p
p_value = poisson.cdf(5, lambda_)
print(f"the resulting p-value is {p_value}")

the resulting p-value is 0.10082956070176977


Abbiamo quindi un p-value di circa 0.101 e quindi possiamo concludere che: 
> Considerando il numero di nazioni all'interno dell'Europa come le nazioni dell'Unione Europea e sotto l'ipotesi di un modello dove ogni nazione ha la stessa probabilita' di costruire un nuovo stadio negli ultimi 20 anni, il fatto che l'Italia ne abbia costruiti di meno non e' statisticamente significativo, ossia non e' inusuale

Riassumendo:
- media attesa: 9.26
- media osservata: 5
- differenza: -4.26
- z-score (sotto approssimazione di Poisson):
$$
z = \frac{5-9.26}{\sqrt{9,26}} = -1.4
$$
- p-value: 0.102

E quindi l'Italia e' sotto la media, ma non in maniera significativa (ossia, dal punto di vista strettamente probabilistico, la spiegazione puo' essere semplicemente data dal caso)

### 1.b., 1.c. Europa Continentale e UEFA

Se parliamo di calcio, e' difficile che ci riferiamo all'Europa considerando solo i paesi che fanno parte dell'Unione Europea, correndo il rischio di non considerare nazioni come l'Inghilterra che, al suo interno, contiene quattro importanti federazione UEFA quali Scozia, Galles, Nord Irlanda e Inghilterra. Usando la stessa metodologia usata nel caso 1.a. 

Come valore di $k$ useremo 44 come Europa Continentale e 55 come UEFA. Usando questi parametri otteniamo

In [ ]:
import pandas as pd

n_countries = [44, 55]
x = 5
mu = []
ita_diff = []
z_score = []
p_value = []

for k in n_countries:
    mu.append(250/k)
    ita_diff.append(x - 250/k)
    z_score.append((x - 250/k) / np.sqrt(250/k))
    p_value.append(poisson.cdf(x, 250/k))

pd.DataFrame(
    {
        "Europa vista come": ['Europa Continentale', 'UEFA'],
        "Numero Nazioni": n_countries,
        "Media": mu,
        "Differenza": ita_diff,
        "z_score": z_score,
        "p_value": p_value
    }
)

,Europa vista come,Numero Nazioni,Media,Differenza,z_score,p_value
0,Europa Continentale,44,5.681818,-0.681818,-0.286039,0.498039
1,UEFA,55,4.545455,0.454545,0.213201,0.695147


Come ci si poteva aspettare, aumentando il numero di nazioni e tenendo costante il numero di nuovi stadi, diminuisce la media per nazione e quindi anche la differenza con il valore osservado di 5 per i nuovi stadi italiani. Di conseguenza, l'affermazione che il numero di nuovi stadi in Italia sia molto minore di quello del resto d'Europa perde di significato statistico. 

Volendo usare un linguaggio statistico un po' piu' rigoroso, se consideriamo 2 ipotesi $H_{0}$ (Null Hypothesis) e $H_{A}$ (Alternative Hypothesis) tali che 
- $H_{0} : \text{L'Italia ha lo stesso numero di nuovi stadi costruiti negli ultimi 20 anni rispetto all'Europa}$:
- $H_{A} : \text{L'Italia ha un numero di nuovi stadi costruiti negli ultimi 20 anni minore rispetto all'Europa}$:

I valori dei p-value ci dicono che non riusciamo a rigettare l'ipotesi nulla e che i dati ci dicono che non esiste una differenza statisticamente significativa tra il valore osservato Italiano e la media Europea nei tre casi considerando le nazioni Europee come a. Solo EU; b. Europa Continentale; c. UEFA.

## Livello 2: Europa, GDP e numero di nuovi stadi

Il Livello 1 considerava la probabilita' di costruire nuovi stadi equamente distribuita lungo tutte le nazioni europee, senza fare nessuna distinzione. Nel Livello 2, proviamo a vedere se inserendo il GDP pro capite nel nostro modello, il fatto che l'Italia abbia costruito solo 5 stadi rispetto ai 245 nel resto d'Europa ci porta a concludere che il paese della pizza, il mandolino, Leonardo Da Vinci, Sinner e del Festival di Sanremo sia in ritardo rispetto alla questione stadi rispetto al continente responsabile di tutte le guerre, carestie, crisi economiche, Trump, le cavallette della storia moderna ed antica. 

### Framework Teorico

Il modello che andremo a considerare per livello 2 ha, come caratteristica principale, l'ipotesi che il numero di nuovi stadi costruito nell'arco degli ultimi 20 anni dipenda dal GDP (o prodotto interno lordo) *pro capite* della nazione dove si costruiscono gli stadi. Dal punto di vista pratico, diciamo che piu' un paese e' ricco, piu' la probabilita' di costruire nuovi stadi sia maggiore rispetto ai paesi con un reddito pro capite inferiore. Per rendere le cose piu' semplici, faremo qualche ulteriore semplificazione:
- Anche se abbiami i dati del GDP pro capite degli ultimi 20 anni, andremo ad usare il dato del solo 2023 (dato consolidato rispetto alle stime per gli anni 2024-25)
- Considerando le nazioni incluse nella lista UEFA, assegneremo all'Inghilterra, Scozia, Galles e Irlanda del Nord il GDP pro capite dell'intero Regno Unito 

Fatte queste precisazioni, possiamo modellare il numero di nuovi stadi per paese come una variabile stocastica con un valore atteso (valore  d'aspettazione o *Expected Value*) proporzionale al GDP pro capite nazionale. Ci sono due modi per costruire questo modello, uno piu' semplice che esegue un'allocazione proporzionale per avere una baseline di confronto tra l'Italia e il resto del mondo, e uno piu' preciso che e' quello di regressione Poissoniana. Con quest'ultimo, possiamo considerare, come nel caso precedente, un *p-value* per decidere se, statisticamente, il numero di 5 nuovi stadi e' significamente diverso (maggiore o minore) rispetto all'Italia.

#### Livello 2 - caso EU, Modello Proporzionale Probabilistico 

Definiamo
- $i \in {1, \dots, N}$ indici dei paesi
- $g_{i}$ e' il GDP *pro capite* del paese $i$
- $S = 250$ e' il numero totale di stadi costruiti in Europa negli ultimi 20 anni
- $p_{i} = \frac{g_{i}}{\sum_{j}g_{j}}$ frazione del GDP *pro capite* per il paese $i$ 

Allora il numero di stadi atteso per ogni paese $i$ sotto le ipotesi di proporzionalita' e':

$$
\mathbb{E}[{Y_{i}}] = S \cdot \frac{g_{i}}{\sum_{j=1}^{N}g_{j}}
$$

e da un un benchmark deterministico. Se, quindi, $y_{i} = 5$ allora possiamo definire che:

- Sopra la media: $y_{i} > \mathbb{E}[{Y_{i}}]$
- In media: $y_{i} \approx \mathbb{E}[{Y_{i}}]$ (entro una determinata tolleranza)
- Sotto la media: $y_{i} < \mathbb{E}[{Y_{i}}]$

Ma con questo modello non c'e' una misura di significanza statistica, ossia non possiamo decidere se, dal punto di vista statistico, valori sopra o sotto la media siano dettati dal caso oppure no.

Vediamo con i valori che estraiamo dai dati, quale puo' essere il nostro benchmark nel caso di ritenere l'Europa come costituita dai soli paesi EU (caso Satana).

In [9]:
from pathlib import Path

data_path = Path('data/processed/EU_countries_GDP.pkl')
EU_countries_GDP = pd.read_pickle(data_path)

# total number of stadiums in the last 20 years

TOTAL_STADIUMS = 250

# Compute expected stadiums proportional to GDP per capita
EU_countries_GDP["GDP_share"] = EU_countries_GDP["GDP"] / EU_countries_GDP["GDP"].sum()
EU_countries_GDP["expected_stadiums"] = TOTAL_STADIUMS * EU_countries_GDP["GDP_share"]

EU_countries_GDP.head()

,Country,Population,GDP,GDP_share,expected_stadiums
0,Austria,9.1,56579.504175,0.048672,12.167905
1,Belgium,11.6,55291.475454,0.047564,11.890903
2,Bulgaria,6.5,15853.208637,0.013637,3.409368
3,Croatia,3.9,22184.227930,0.019084,4.770907
4,Cyprus,1.3,36623.691406,0.031505,7.876237


In [6]:
EU_countries_GDP[EU_countries_GDP['Country'] == 'Italy']

,Country,Population,GDP,GDP_share,expected_stadiums
14,Italy,59.2,39277.083878,0.033787,8.446872


Quindi nel caso in cui teniamo in considerazione solo i paesi dell'Unione Europea e il loro GDP *pro capite* e supponendo che negli ultimi 20 anni sono stati costruiti 250 stadi nell'EU, ci aspetteremmo circa 8.4 stadi nuovi in Italia. Confrontando il valore atteso (8.4) con quello osservato (5), possiamo dire che il valore osservato e' al di sotto delle aspettative (mancano quasi 3 stadi e mezzo. 3 stadi, una curva, e un distinto superiore o inferiore).

Questo modello, pero', non ci dice se questa discrepanza e' statisticamente significativa, ossia se potrebbe essere dettata dal caso (fluttuazione statistica) oppure c'e' qualche potere forte che cospira contro la costruzione di nuovi stadi (in questo caso potremmo addirittura parlare di massoni che si oppongono a costruizioni, una lotta fratricida insomma). 

Per avere un quadro statisticamente piu' preciso, utilizziamo un modello di Poisson (modello stocastico con inferenza statistica).

#### Livello 2: Caso Satana (EU), Modello Stocastico Poissoniano

Il modello poissoniano e' particolarmente adatto per valutare il numero di eventi in un determinato intervallo temporale (si usa una binomiale negativa o di Pascal se e' piu' dispersa, ossia ha una varianza maggiore).

In questo caso supponiamo che il numero di stadi nel tempo e' modellato attraverso una distribuzione di Poisson:

$$
Y_{i} = \text{Poisson}(\lambda_{i})
$$

con 
- $\lambda_{i} = \alpha \cdot g_{i}$

Per calcolare $\alpha$ usiamo il fatto che il numero totale di stadi negli ultimi 20 anni e' stato, secondo Marotta, 250, per cui:

$$
\sum_{i} \lambda_{i} = 250 = \alpha \sum_{i} g_{i}
$$

con, ancora, $g_{i}$ il GDP *pro capite* del paese $i$. Quindi la stima di $\alpha$ e'

$$
\alpha = \frac{250}{\sum_{i} g_{i}}
$$

e quindi

$$
\lambda_{i} = 250 \cdot \frac{g_{i}}{\sum_{j}g_{j}}
$$

I lettori piu' attenti si saranno accorti che il risultato finale e' uguale al modello deterministico, ma, in questo caso, abbiamo una ***distribuzione*** che ci permette una valutazione statistica. 

#### Hypothesis Test per una nazione

- valore osservato: $y_{i}$
- valore atteso: $\lambda_{i}$
- ipotesi nulla: $H_{0} : Y_{i} \sim \text{Poisson}(\lambda_{i})$


> ATTENZIONE, PARTE DA RISCRIVERE. NON SO COME TRADURRE LEFT-TAIL, RIGHT-TAIL
Se ora calcoliamo:

- $P(Y_{i} \leq 5 | \lambda_{i})$ e' significativamente sotto il valore atteso dal punto di vista statistico se il p-value $< \alpha$ (in questo caso $\alpha$ e' il *significance level* (pare in italiano si dica livello di significativita'))
- $P(Y_{i} \geq 5 | \lambda_{i})$ e' significativamente sotto il valore atteso dal punto di vista statistico se il p-value $< \alpha$ (in questo caso $\alpha$ e' il *significance level* (pare in italiano si dica livello di significativita'))

In [10]:
from scipy.stats import poisson

# select Italy's row
row = EU_countries_GDP[EU_countries_GDP['Country'] == 'Italy'].iloc[0]

observed = 5
expected = row["expected_stadiums"]

# compute p-values
p_left = poisson.cdf(observed, expected)          # P(Y ≤ observed)
p_right = 1 - poisson.cdf(observed - 1, expected) # P(Y ≥ observed)
# p_two_sided = 2 * min(p_left, p_right)
# 2023.03.05 Fix: a more proper two-sided p-value for discrete distributions:
p_two_sided = min(1, 2 * min(p_left, p_right))

print("Country:", row["Country"])
print("Expected:", expected)
print("Observed:", observed)
print("p_left:", p_left)
print("p_right:", p_right)
print("p_two_sided:", p_two_sided)

Country: Italy
Expected: 8.446871829845312
Observed: 5
p_left: 0.1536382269624755
p_right: 0.9232515233913667
p_two_sided: 0.307276453924951


Quello che ci interessa, in questo caso, e' il p-value della coda della parte sinistra (il `p_left`) che e' pari a circa 0.15. In questo caso, se scegliamo il solito significant level di 0.05, il p-value e' maggiore del signiicance level $\alpha$ per cui 

> NON possiamo escludere che in realta' il valore di 5 nuovi stadi in Italia, non sia in linea con la media europea e che la differenza con il benchmark che abbiamo trovato nel caso deterministico, puo' essere dovuta a una "fluttuazione statistica" all'interno della variabilita' di ogni nazione dell'EU (Satana). 

#### Livello 2 - casi Europa Dei Popoli (quella vera!!1!!) e UEFA (ladri corrotti e, fondamentalmente, contro la mia squadra qualunque sia la mia squadra). 

Avendo discusso il framework teorico nelle sezioni precedenti, cerchiamo ora di rispondere alla domanda se l'Italia rappresenti veramente il fanalino di coda dell'Europa considerando prima un'Europa Continentale (quella dei popoli sovrani, LE RADICI!!!11!!) e poi l'UEFA (quella delle elite corrotte che ci stanno portando via il calcio vero, quello di una volta quando contavano i sentimenti e non il vile denaro). Il parametro che usiamo e' di nuovo il prodotto interno lordo *pro capite*, quello che cambia e' il numero dei paesi coinvolti. Come anticipato, abbiamo fatto alcune semplificazioni per quanto riguarda l'UEFA e assegnato al GDP di Inghilterra, Galles, Scozia e Irlanda del Nord quello del Regno Unito, senza fare distinzioni tra le diverse regioni, anche se in realta' le diverse zone del Regno Unito hanno un reddito *pro capite* diverso. Nel caso dell'UEFA, quindi, andremo a calcolare il numero di nuovi stadi negli ultimi 20 anni **per federazione** e non per nazione. Se sembriamo pedanti, prendetevela con Il Post che ci costringe a spiegare bene ogni passaggio altrimenti arrivano quelli della redazione con il ditino alzato borbottando "ah ah ah mmmmh"(cit.).  

Per entrambi i casi che riguardano l'Europa Continentale e l'UEFA, calcoliamo prima il valore di benchmark (il valore atteso) e poi la significanza stastistica del discostamento dal valore atteso. 

- Valori attesi

In [12]:
from pathlib import Path

data_path_continental_europe = Path('data/processed/geographic_europe_GDP.pkl')
data_path_UEFA = Path('data/processed/UEFA_countries_GDP.pkl')

geographic_europe_GDP = pd.read_pickle(data_path_continental_europe)
UEFA_countries_GDP = pd.read_pickle(data_path_UEFA)

# total number of stadiums in the last 20 years

TOTAL_STADIUMS = 250

# Compute expected stadiums proportional to GDP per capita for both geographic Europe and UEFA countries

geographic_europe_GDP["GDP_share"] = geographic_europe_GDP["GDP"] / geographic_europe_GDP["GDP"].sum()
geographic_europe_GDP["expected_stadiums"] = TOTAL_STADIUMS * geographic_europe_GDP["GDP_share"]

UEFA_countries_GDP["GDP_share"] = UEFA_countries_GDP["GDP"] / UEFA_countries_GDP["GDP"].sum()
UEFA_countries_GDP["expected_stadiums"] = TOTAL_STADIUMS * UEFA_countries_GDP["GDP_share"]

print(geographic_europe_GDP.head(3))
print('---')
print(UEFA_countries_GDP.head(3))


   Country  Population            GDP  GDP_share  expected_stadiums
0  Albania         2.80   9730.869219   0.004110           1.027471
1  Andorra         0.08  46812.426101   0.019771           4.942867
2  Armenia         2.80   8159.087210   0.003446           0.861508
---
   Country  Population            GDP  GDP_share  expected_stadiums
0  Albania         2.80   9730.869219   0.003799           0.949681
1  Andorra         0.08  46812.426101   0.018275           4.568642
2  Armenia         2.80   8159.087210   0.003185           0.796283


In [16]:
print(geographic_europe_GDP[geographic_europe_GDP['Country'] == 'Italy'])
print('---')
print(UEFA_countries_GDP[UEFA_countries_GDP['Country'] == 'Italy'])


   Country  Population            GDP  GDP_share  expected_stadiums
24   Italy         59.2  39277.083878   0.016589           4.147219
---
   Country  Population            GDP  GDP_share  expected_stadiums
25   Italy         59.2  39277.083878   0.015333           3.833233


Come nel caso di una distribuzione random equiprobabilistica, appena consideriamo tutti i paesi dell'Europa continentale e UEFA, il valore atteso per gli stadi nuovi scende. Sorprendentemente, il valore atteso, ossia quello che dovremmo aspettarci quando ipotizziamo che la costruizione di nuovi stadi dipenda dal prodotto interno lordo, e' inferiore a quello osservato, che per Marotta e' basso. Se dovessimo fidarci del modello, dovremmo concludere che costruire 5 nuovi stadi significa essere al di sopra della media europea, e quindi altro che fanalini di coda.

Perche' abbiamo scritto "sorprendentemente"? Perche' in effetti il GDP dell'Italia dovrebbe essere superiore alla media europea, e quindi il numero atteso di stadi ce lo aspettiamo piu' alto. Vediamo se i dati ci danno un'idea del perche' otteniamo questi risultati.  

In [17]:
geographic_europe_GDP.sort_values(by="expected_stadiums", ascending=False)[:10]

,Country,Population,GDP,GDP_share,expected_stadiums
33,Monaco,0.04,256799.788613,0.108461,27.115176
28,Liechtenstein,0.04,206780.590353,0.087335,21.833710
30,Luxembourg,0.70,133230.619179,0.056271,14.067658
23,Ireland,5.10,106818.917131,0.045116,11.278879
48,Switzerland,8.70,100623.549627,0.042499,10.624718
19,Gibraltar,0.03,100603.000000,0.042490,10.622548
37,Norway,5.50,87497.217965,0.036955,9.238724
22,Iceland,0.40,82138.789297,0.034692,8.672934
14,Faroe Islands,0.05,72465.376966,0.030606,7.651531
12,Denmark,5.90,68043.546697,0.028739,7.184635


E in effetti, come i piu' attenti lettori avranno ipotizzato, ci sono diversi piccoli paesi, quali il principato di Monaco, Liechtenstein, Lussemburgo, Gibilterra, Isole Faroe, hanno un numero atteso di stadi del tutto irragionevole dovuto al loro alto prodotto interno lordo *pro capite*, se non altro considerando la loro estensione e popolazione. Questo, dal punto di vista statistico, allarga a dismisura la distribuzione e comprime verso il basso il valore di aspettazione di nazioni piu' popolose.

Era un effetto facilmente preventivabile, che ogni bravo analista puo' prevedere in anticipo. In questo articolo, lo abbiamo lasciato fino a questo punto anche per sottolineare uno dei punti principali di questa discussione: piu' le dichiarazioni sono vaghe, poco circostanziate, piu' i risultati sono inaffidabili. Questo vale anche quando si sceglie un modello per descrivere la realta' (e prevedere possibili scenari). Quando si sceglie un modello, bisogna essere coscienti del fatto che un modello, per sua natura, ha dei limiti e rappresenta un'approssimazione della realta'. Quando si fanno questo tipo di analisi, si cerca sempre di favorire il modello "piu' economico", ossia quel modello che cerca di tenere in considerazione meno variabili possibili ma che siano quelle che pesano di piu' nella valutazione finale. La ragione per la quale si favoriscono i modelli "piu' economici" e' non solo che computazionalmente piu' leggeri e facilmente maneggiabili (possono essere debuggati piu' facilmente), ma anche perche' quando si devono "spiegare", sono facilmente comprensibili. Trovare l'equilibrio 

#### References
- GDP per Capita: [World Bank Group](https://data.worldbank.org/indicator/NY.GDP.PCAP.CD)